In [14]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from src.logger import logging
import os
import time
import mlflow
import scipy.sparse
import dagshub



In [15]:
df = pd.read_csv(r'D:\Resume-Screening-Matching-System\data_upload_S3\data\cleaned_dataset.csv')
df.head()

,Role,Resume
0,E-commerce Specialist,Here's a professional resume for Jason Jones:\...
1,Game Developer,Here's a professional resume for Ann Marshall:...
2,Human Resources Specialist,Here's a professional resume for Patrick Mccla...
3,E-commerce Specialist,Here's a professional resume for Patricia Gray...
4,E-commerce Specialist,Here's a professional resume for Amanda Gross:...


Preprocessing Data:

In [16]:
def clean_text(text):
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove URLs (linkedin, github, websites)
    text = re.sub(r'http\S+|www\.\S+|linkedin\.com/\S+|github\.com/\S+', ' ', text)

    # Remove phone numbers
    text = re.sub(
        r'(\+\d{1,3}[-.\s]?)?(\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4})',
        ' ',
        text
    )

    # Remove markdown links
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)

    # Remove bullet symbols
    text = re.sub(r'[•▪◦●■♦★]', ' ', text)

    # Remove asterisks used as bullets
    text = re.sub(r'\*', ' ', text)

    # Keep only letters, numbers, +, #, /, . and spaces
    text = re.sub(r'[^a-zA-Z0-9+#/. ]', ' ', text)

    # Convert to lowercase
    text = text.lower()

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    

    return text

def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def normalize_text(text):
    """Normalize the text by applying all preprocessing steps."""
    text = clean_text(text)
    text = lemmatization(text)
    text = remove_stop_words(text)
    return text

In [17]:
df['Resume'] = df['Resume'].apply(lambda x: normalize_text(x))

In [18]:
df['Resume'][0]

'professional resume jason jones jason jones e commerce specialist contact information email phone linkedin summary result driven e commerce specialist 5+ year experience inventory management seo online advertising analytics. proven track record increasing online sale improving website traffic optimizing inventory levels. skilled analyzing complex data set identifying trend making data driven decisions. passionate staying date latest e commerce trend technologies. professional experience e commerce specialist xyz corporation 2018 present managed inventory level across multiple channel resulting 25 reduction stockouts 15 reduction overstocking developed implemented seo strategy increased website traffic 30 improved search engine ranking 20 created executed online advertising campaign generated 50 increase sale 20 increase conversion rate analyzed website analytics identify trend optimize user experience improve customer engagement collaborated cross functional team launch new product li

Target Column into Categorical values

In [19]:
label = LabelEncoder()
df['Role'] = label.fit_transform(df['Role'])
df.head()

,Role,Resume
0,15,professional resume jason jones jason jones e ...
1,17,professional resume ann marshall ann marshall ...
2,18,professional resume patrick mcclain patrick mc...
3,15,professional resume patricia gray patricia gra...
4,15,professional resume amanda gross amanda gross ...


In [20]:
VECTORIZERS = {
    'BoW': CountVectorizer(max_features=200, ngram_range=(1,2)),
    'TF-IDF': TfidfVectorizer(max_features=200, ngram_range=(1,2))
}

ALGORITHMS = {
    'LogisticRegression': LogisticRegression(),
    'MultinomialNB': MultinomialNB(),
    'RandomForest': RandomForestClassifier(),
    'LinearSVC': LinearSVC()
}


In [21]:
def log_model_params(algo_name, model):
    """Logs hyperparameters of the trained model to MLflow."""
    params_to_log = {}
    if algo_name == 'LogisticRegression':
        params_to_log["C"] = model.C
    elif algo_name == 'MultinomialNB':
        params_to_log["alpha"] = model.alpha
    elif algo_name == 'RandomForest':
        params_to_log["n_estimators"] = model.n_estimators
        params_to_log["max_depth"] = model.max_depth
    elif algo_name == 'LinearSVC':
        params_to_log["C"] = model.C
    

    mlflow.log_params(params_to_log)

In [23]:
CONFIG = {
    "data_path": "notebooks/data.csv",
    "test_size": 0.2,
    "mlflow_tracking_uri": "https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow",
    "dagshub_repo_owner": "mdzaid382",
    "dagshub_repo_name": "Resume-Screening-Matching-System",
    "experiment_name": "Bow-vs-TfIdf"
}

mlflow.set_tracking_uri(CONFIG["mlflow_tracking_uri"])
dagshub.init(repo_owner=CONFIG["dagshub_repo_owner"], repo_name=CONFIG["dagshub_repo_name"], mlflow=True)
mlflow.set_experiment(CONFIG["experiment_name"])

with mlflow.start_run(run_name="All Experiments") as parent_run:
    for algo_name, algorithm in ALGORITHMS.items():
        for vec_name, vectorizer in VECTORIZERS.items():
            with mlflow.start_run(run_name=f"{algo_name} with {vec_name}", nested=True) as child_run:
                try:
                    
                    logging.info("Initializing train test split...")
                    X_train, X_test, y_train, y_test = train_test_split(df['Resume'], df['Role'], test_size=CONFIG["test_size"], random_state=42)

                    logging.info("Vectorizing text data...")
                    X_train = vectorizer.fit_transform(X_train)
                    X_test = vectorizer.transform(X_test)
                    
                    logging.info("Logging preprocessing parameters...")
                    mlflow.log_params({
                        "vectorizer": vec_name,
                        "algorithm": algo_name,
                        "test_size": CONFIG["test_size"]
                    })

                    logging.info("Training model...")
                    model = algorithm
                    model.fit(X_train, y_train)

                    logging.info("Logging model parameters...")
                    log_model_params(algo_name, model)
                   
                    logging.info("Evaluating model...")
                    y_pred = model.predict(X_test)

                    metrics = {
                        "accuracy": accuracy_score(y_test, y_pred,),
                        "precision": precision_score(y_test, y_pred, average='weighted'),
                        "recall": recall_score(y_test, y_pred, average='weighted'),
                        "f1_score": f1_score(y_test, y_pred, average='weighted')
                    }

                    logging.info("Logging evaluation metrics...")
                    mlflow.log_metrics(metrics)

                    logging.info("Logging model artifact...")
                    input_example = X_test[:5] if not scipy.sparse.issparse(X_test) else X_test[:5].toarray()
                    mlflow.sklearn.log_model(model, "model", input_example=input_example)
                    # Print results for verification
                    print(f"\nAlgorithm: {algo_name}, Vectorizer: {vec_name}")
                    print(f"Metrics: {metrics}")

                except Exception as e:
                    print(f"Error in training {algo_name} with {vec_name}: {e}")
                    mlflow.log_param("error", str(e))




[ 2026-06-03 15:19:04,834 ] httpx - INFO - HTTP Request: GET https://dagshub.com/api/v1/repos/mdzaid382/Resume-Screening-Matching-System "HTTP/1.1 200 OK"


Initialized MLflow to track repo "mdzaid382/Resume-Screening-Matching-System"

[ 2026-06-03 15:19:04,839 ] dagshub - INFO - Initialized MLflow to track repo "mdzaid382/Resume-Screening-Matching-System"


Repository mdzaid382/Resume-Screening-Matching-System initialized!

[ 2026-06-03 15:19:04,841 ] dagshub - INFO - Repository mdzaid382/Resume-Screening-Matching-System initialized!


...


Algorithm: MultinomialNB, Vectorizer: TF-IDF
Metrics: {'accuracy': 0.9533169533169533, 'precision': 0.9578191814199982, 'recall': 0.9533169533169533, 'f1_score': 0.9464725541088086}
🏃 View run MultinomialNB with TF-IDF at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/2/runs/3cfb6bb96b0f4ea5bbb4bcfde619758e
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/2
[ 2026-06-03 15:20:41,640 ] root - INFO - Initializing train test split...
[ 2026-06-03 15:20:41,644 ] root - INFO - Vectorizing text data...
[ 2026-06-03 15:20:45,568 ] root - INFO - Logging preprocessing parameters...
[ 2026-06-03 15:20:46,043 ] root - INFO - Training model...
[ 2026-06-03 15:20:53,434 ] root - INFO - Logging model parameters...
[ 2026-06-03 15:20:53,973 ] root - INFO - Evaluating model...
[ 2026-06-03 15:20:54,038 ] root - INFO - Logging evaluation metrics...
[ 2026-06-03 15:20:54,577 ] root - INFO - Logging model arti


Algorithm: RandomForest, Vectorizer: BoW
Metrics: {'accuracy': 0.9941031941031941, 'precision': 0.994163090583124, 'recall': 0.9941031941031941, 'f1_score': 0.9941049829903837}
🏃 View run RandomForest with BoW at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/2/runs/41d409cae4d344878451dff68970a242
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/2
[ 2026-06-03 15:22:05,977 ] root - INFO - Initializing train test split...
[ 2026-06-03 15:22:05,981 ] root - INFO - Vectorizing text data...
[ 2026-06-03 15:22:09,776 ] root - INFO - Logging preprocessing parameters...
[ 2026-06-03 15:22:10,218 ] root - INFO - Training model...
[ 2026-06-03 15:22:21,288 ] root - INFO - Logging model parameters...
[ 2026-06-03 15:22:21,789 ] root - INFO - Evaluating model...
[ 2026-06-03 15:22:21,854 ] root - INFO - Logging evaluation metrics...
[ 2026-06-03 15:22:22,301 ] root - INFO - Logging model artifact...



Algorithm: RandomForest, Vectorizer: TF-IDF
Metrics: {'accuracy': 0.9936117936117936, 'precision': 0.9939361127827967, 'recall': 0.9936117936117936, 'f1_score': 0.9935877836006017}
🏃 View run RandomForest with TF-IDF at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/2/runs/93b68d4eccd04384a072df328c47cbd1
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/2
[ 2026-06-03 15:23:22,703 ] root - INFO - Initializing train test split...
[ 2026-06-03 15:23:22,708 ] root - INFO - Vectorizing text data...
[ 2026-06-03 15:23:26,661 ] root - INFO - Logging preprocessing parameters...
[ 2026-06-03 15:23:27,223 ] root - INFO - Training model...
[ 2026-06-03 15:23:32,130 ] root - INFO - Logging model parameters...
[ 2026-06-03 15:23:32,628 ] root - INFO - Evaluating model...
[ 2026-06-03 15:23:32,640 ] root - INFO - Logging evaluation metrics...
[ 2026-06-03 15:23:33,163 ] root - INFO - Logging model artifa


Algorithm: LinearSVC, Vectorizer: BoW
Metrics: {'accuracy': 0.9921375921375921, 'precision': 0.9922931850158989, 'recall': 0.9921375921375921, 'f1_score': 0.9921321772520943}
🏃 View run LinearSVC with BoW at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/2/runs/4da6f4d4cb4a4732af604097127aebce
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/2
[ 2026-06-03 15:23:45,146 ] root - INFO - Initializing train test split...
[ 2026-06-03 15:23:45,149 ] root - INFO - Vectorizing text data...
[ 2026-06-03 15:23:49,072 ] root - INFO - Logging preprocessing parameters...
[ 2026-06-03 15:23:49,650 ] root - INFO - Training model...
[ 2026-06-03 15:23:50,895 ] root - INFO - Logging model parameters...
[ 2026-06-03 15:23:51,391 ] root - INFO - Evaluating model...
[ 2026-06-03 15:23:51,401 ] root - INFO - Logging evaluation metrics...
[ 2026-06-03 15:23:51,903 ] root - INFO - Logging model artifact...



Algorithm: LinearSVC, Vectorizer: TF-IDF
Metrics: {'accuracy': 0.9945945945945946, 'precision': 0.994780600776113, 'recall': 0.9945945945945946, 'f1_score': 0.9945885413146585}
🏃 View run LinearSVC with TF-IDF at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/2/runs/55e88fd25e8d458da3ebf74ebb6363c3
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/2
🏃 View run All Experiments at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/2/runs/5d9c60cc5c564eebae080a792df1ace9
🧪 View experiment at: https://dagshub.com/mdzaid382/Resume-Screening-Matching-System.mlflow/#/experiments/2
